# Checkout Conversion Funnel — Part 1: Funnel Drop-off Analysis

**Business context:** A BNPL provider earns revenue only on completed purchases. Understanding where borrowers exit the funnel — and why — is the first step to improving checkout conversion.

This notebook answers:
1. What is the end-to-end conversion rate from page load to purchase?
2. Where is the biggest drop-off?
3. How do conversion rates differ by merchant vertical and risk tier?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import subprocess, sys, os

plt.style.use('seaborn-v0_8-whitegrid')

# Generate data if not already present
data_dir = '../data'
if not os.path.exists(f'{data_dir}/funnel_events.csv'):
    subprocess.run([sys.executable, f'{data_dir}/generate_data.py'], cwd=data_dir)

funnel   = pd.read_csv(f'{data_dir}/funnel_events.csv', parse_dates=['session_date'])
merchants = pd.read_csv(f'{data_dir}/merchant_summary.csv')

funnel = funnel.merge(merchants[['merchant_id', 'vertical']], on='merchant_id', how='left')
print(f'Sessions loaded: {len(funnel):,}')

## 1. Overall Funnel Waterfall

In [ ]:
stages = [
    ('reached_page_load',         'Page Load'),
    ('passed_bnpl_click',         'BNPL Click'),
    ('passed_form_start',         'Form Start'),
    ('passed_form_complete',      'Form Complete'),
    ('passed_approved',           'Credit Approved'),
    ('passed_purchase_complete',  'Purchase Complete'),
]

stage_counts = [(label, funnel[col].sum()) for col, label in stages]
labels = [s[0] for s in stage_counts]
counts = [s[1] for s in stage_counts]
pct_of_top = [c / counts[0] * 100 for c in counts]
pct_prev   = [100.0] + [counts[i] / counts[i-1] * 100 for i in range(1, len(counts))]

summary = pd.DataFrame({
    'Stage': labels,
    'Count': counts,
    '% of Page Loads': [f'{p:.1f}%' for p in pct_of_top],
    'Step Conversion': [f'{p:.1f}%' for p in pct_prev]
})
print(summary.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#3498db' if i != 4 else '#e74c3c' for i in range(len(labels))]
bars = ax.bar(labels, pct_of_top, color=colors, edgecolor='white', width=0.6)
for bar, pct, prev in zip(bars, pct_of_top, pct_prev):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{pct:.1f}%\n({prev:.0f}% step)', ha='center', va='bottom', fontsize=8.5)
ax.set_ylabel('% of Page Loads Reaching Stage')
ax.set_title('BNPL Checkout Funnel — Overall Conversion', fontsize=13)
ax.set_ylim(0, 115)
ax.tick_params(axis='x', rotation=15)

# Annotate the biggest drop
ax.annotate('← Largest drop-off:\nCredit decision',
            xy=(4, pct_of_top[4]), xytext=(4.3, pct_of_top[4] + 12),
            arrowprops=dict(arrowstyle='->', color='#e74c3c'),
            color='#e74c3c', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/figures/funnel_waterfall.png', dpi=150)
plt.show()

## 2. Conversion by Merchant Vertical

In [ ]:
vertical_conv = funnel.groupby('vertical').agg(
    sessions=('reached_page_load', 'sum'),
    purchases=('passed_purchase_complete', 'sum')
).assign(
    conversion_rate=lambda x: x['purchases'] / x['sessions'] * 100
).sort_values('conversion_rate', ascending=True).reset_index()

overall_conv = funnel['passed_purchase_complete'].sum() / len(funnel) * 100

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(vertical_conv['vertical'], vertical_conv['conversion_rate'],
               color='#3498db', edgecolor='white')
ax.axvline(overall_conv, color='#e74c3c', linestyle='--',
           label=f'Overall avg: {overall_conv:.1f}%')
ax.set_xlabel('End-to-End Conversion Rate (%)')
ax.set_title('Checkout Conversion by Merchant Vertical', fontsize=12)
ax.legend()
for bar, rate in zip(bars, vertical_conv['conversion_rate']):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f'{rate:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../outputs/figures/conversion_by_vertical.png', dpi=150)
plt.show()

## 3. Approval Rate and Conversion by Risk Tier

In [ ]:
tier_stats = funnel.groupby('risk_tier').agg(
    sessions=('reached_page_load', 'sum'),
    approvals=('passed_approved', 'sum'),
    purchases=('passed_purchase_complete', 'sum')
).assign(
    approval_rate=lambda x: x['approvals'] / x['sessions'] * 100,
    conversion_rate=lambda x: x['purchases'] / x['sessions'] * 100
).reset_index()

tier_order = ['Low', 'Mid', 'High']
tier_stats['risk_tier'] = pd.Categorical(tier_stats['risk_tier'], categories=tier_order, ordered=True)
tier_stats = tier_stats.sort_values('risk_tier')

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(tier_stats))
w = 0.35
b1 = ax.bar(x - w/2, tier_stats['approval_rate'], w, label='Approval Rate', color='#3498db')
b2 = ax.bar(x + w/2, tier_stats['conversion_rate'], w, label='Conversion Rate', color='#2ecc71')
ax.set_xticks(x)
ax.set_xticklabels([f'{t} Risk' for t in tier_stats['risk_tier']])
ax.set_ylabel('Rate (%)')
ax.set_title('Approval Rate vs. Conversion Rate by Risk Tier', fontsize=12)
ax.legend()
for bars in [b1, b2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('../outputs/figures/approval_conversion_by_tier.png', dpi=150)
plt.show()

print('\nKey insight: The gap between approval rate and conversion rate')
print('reveals post-approval drop-off — borrowers who were approved but did not complete purchase.')
print(tier_stats[['risk_tier','approval_rate','conversion_rate']].to_string(index=False))